# Ingest Fact Raw Data into Bronze Layer

This notebook ingests raw order items CSV data from the source volume into the bronze layer of the e-commerce medallion architecture. It defines the schema, reads all CSV files with metadata columns for lineage tracking, and writes the raw data as a managed Delta table in the bronze schema.

## Source and Target

| Source | Target | Rows |
| --- | --- | --- |
| `/Volumes/ecommerce/source_data/raw/order_items/landing/*.csv` | `ecommerce.bronze.brz_order_items` | 183,378 |

## Schema (13 data columns, all `StringType`)

All columns are read as `StringType` to preserve raw data as-is during bronze ingestion. Type casting and data cleaning will be performed in the silver layer.

| Column | Description |
| --- | --- |
| `dt` | Transaction date (string) |
| `order_ts` | Order timestamp (string) |
| `customer_id` | Customer identifier |
| `order_id` | Order identifier |
| `item_seq` | Item sequence number within an order |
| `product_id` | Product identifier |
| `quantity` | Quantity ordered (may contain non-numeric values) |
| `unit_price_currency` | Currency code for unit price |
| `unit_price` | Unit price amount |
| `discount_pct` | Discount percentage |
| `tax_amount` | Tax amount |
| `channel` | Sales channel (e.g., app, web) |
| `coupon_code` | Coupon code applied (may be NULL) |

Two metadata columns are added during ingestion:

* `_source_file` — file path each row was ingested from (from Spark's `_metadata.file_path`)
* `_ingested_at` — timestamp marking when the data was loaded

## Processing Steps

1. **Define schema** — explicit `StructType` with all `StringType` fields
2. **Read CSV** — `spark.read` with `header=True`, `delimiter=","`, and the defined schema (no schema inference)
3. **Add metadata** — `_source_file` and `_ingested_at` for lineage tracking
4. **Write to Bronze** — `saveAsTable` as a managed Delta table with `mode("overwrite")` and `mergeSchema=True`

In [0]:
# Import required libraries for schema definition and DataFrame operations
from pyspark.sql.types import StructField, StructType, StringType, IntegerType, DateType, BooleanType
import pyspark.sql.functions as F

In [0]:
# Define the catalog name for the e-commerce project
catalog_name = 'ecommerce'

# Define the path to the raw order items CSV files in the source volume
raw_file_path = '/Volumes/ecommerce/source_data/raw/order_items/landing/*.csv'

In [0]:
# Define the schema for the order items CSV file.
# All columns are read as StringType to preserve raw data as-is during bronze ingestion.
# Type casting and data cleaning will be performed in the silver layer.

order_items_schema = StructType(
    [
        StructField("dt", StringType(), True),
        StructField("order_ts", StringType(), True),
        StructField("customer_id", StringType(), True),
        StructField("order_id", StringType(), True),
        StructField("item_seq", StringType(), True),
        StructField("product_id", StringType(), True),
        StructField("quantity", StringType(), True),
        StructField("unit_price_currency", StringType(), True),
        StructField("unit_price", StringType(), True),
        StructField("discount_pct", StringType(), True),
        StructField("tax_amount", StringType(), True),
        StructField("channel", StringType(), True),
        StructField("coupon_code", StringType(), True),
    ]
)

In [0]:
# Read all CSV files from the landing path using the defined schema.
# Add metadata columns: _source_file (file path) and _ingested_at (timestamp) for lineage tracking.

df_orders = spark.read \
                .option("header", True) \
                .option("delimiter", ",") \
                .schema(order_items_schema) \
                .csv(raw_file_path) \
                .withColumn("_source_file", F.col("_metadata.file_path")) \
                .withColumn("_ingested_at", F.current_timestamp()
)

# table preview
df_orders.show(10, truncate=False)

In [0]:
# Verify the row and column counts of the loaded DataFrame
print(f"\nRow count: {df_orders.count()}")
print(f"Column count: {len(df_orders.columns)}\n")


**Save the raw order items data to the Bronze layer**

The raw CSV data has been loaded into a DataFrame with the following metadata columns added:

- **`_source_file`** — The file path each row was ingested from, for lineage tracking.
- **`_ingested_at`** — The timestamp when each row was loaded into the DataFrame.

All columns are stored as `StringType` to preserve the raw data as-is. The DataFrame is now ready to be written to the bronze layer as a managed Delta table (`brz_order_items`).

In [0]:
# Write the raw order items data to the bronze layer as a managed Delta table

df_orders \
    .write \
    .mode("overwrite") \
    .format("delta") \
    .option("mergeSchema", True) \
    .saveAsTable(f"{catalog_name}.bronze.brz_order_items")

## Summary

The raw order items data has been ingested into the Bronze layer:

| Property | Value |
| --- | --- |
| Bronze Table | `ecommerce.bronze.brz_order_items` |
| Source | `/Volumes/ecommerce/source_data/raw/order_items/landing/*.csv` |
| Total Rows | 183,378 |
| Total Columns | 15 (13 data + 2 metadata) |
| Write Mode | `overwrite` with `mergeSchema=True` |

All columns are stored as `StringType` at this stage. Type casting, data cleaning, and enrichment will happen in the Silver and Gold layers.